# 03 — Action repeat (HOOK B)

The temporal intervention: call the model **half as often** and hold each
action for two environment steps instead of one.

```python
actions = np.repeat(actions, 2, axis=0)
```

That single line is the whole mechanism. What follows is why it goes where
it goes, how it differs from chunk execution, and how to measure it without
reporting the wrong number.

## Action repeat vs chunk execution — these are different interventions

Both reduce the number of model calls. They are not interchangeable.

| | action repeat 2 | chunk-exec 2 |
|---|---|---|
| what gets executed | the **same action, copied** | **two different actions** the model actually predicted |
| requires | nothing | the policy must emit multiple actions per call |
| information lost | yes — motion becomes stepwise | comparatively little |
| works on a single-action policy | ✅ | ❌ **not defined** |

**A policy that emits one action per call has no chunk to truncate**, so
chunk-exec is not a worse option for it — it does not exist. Action repeat
is therefore the only temporal intervention that runs *identically*
regardless of whether a policy chunks, which is what makes it comparable
across a set of backbones that do not all chunk.

That is itself a finding worth stating in the analysis: the temporal axis
is really two mechanisms — one universal but lossy, one lossless but
requiring native action chunking — and a backbone that cannot chunk is
forced onto the worse one.

In our own runs the two came out in opposite directions on different
backbones — chunk-exec at k=2 was one backbone's best result (+13.6 at 1.9×
faster) and cost another 12.5 points. That is context, not a prediction:
see the appendix at the end.

## HOOK B — where this goes

On the **action array the policy returned, before it reaches `env.step`.**

```
actions = policy.step(image, instruction)   # (T, action_dim)
actions = np.repeat(actions, 2, axis=0)     ← HERE   -> (2T, action_dim)
for row in actions:
    env.step(row)
```

### Order matters if combined with chunk truncation

If a run uses both, truncate **first**, then repeat:

```python
actions = actions[:k]                       # chunk-exec
actions = np.repeat(actions, r, axis=0)     # action repeat
```

Repeating first and then truncating silently produces a different
condition — with `k=2, r=2` it would execute the first action twice and
nothing else, rather than two actions twice each.

In [ ]:
import numpy as np


def apply_action_repeat(actions, repeat):
    """Hold each action for `repeat` consecutive environment steps.

    np.repeat (not np.tile): repeat=2 on [a, b, c] gives [a, a, b, b, c, c],
    whereas tile would give [a, b, c, a, b, c] — a completely different
    trajectory that would still run and still produce a number.
    """
    actions = np.asarray(actions)
    if repeat <= 1:
        return actions
    return np.repeat(actions, int(repeat), axis=0)


def make_action_repeat_hook(repeat=2, exec_chunk=0):
    """Returns an action_fn for `run_episode` in notebook 01."""

    def action_fn(actions, state):
        if exec_chunk > 0:
            actions = actions[:exec_chunk]      # truncate first
        return apply_action_repeat(actions, repeat)

    return action_fn


# usage with the loop from 01:
#   run_episode(adapter, policy, instruction,
#               action_fn=make_action_repeat_hook(repeat=2))

In [ ]:
demo = np.array([[0.1, 0.0], [0.2, 0.0], [0.3, 0.0]])
print("original      :", demo[:, 0].tolist())
print("repeat 2      :", apply_action_repeat(demo, 2)[:, 0].tolist())
print("tile (WRONG)  :", np.tile(demo, (2, 1))[:, 0].tolist())

## How this was wired into the three backbones we ran

Also nowhere — same as hook A. One line in the shared loop, after every
policy has already returned. From `adaptive_sparse_vla/eval_libero.py`:

```python
action_chunk = model.step(policy_image, instruction, wrist_image=policy_wrist)

if args.exec_chunk > 0:
    action_chunk = action_chunk[: args.exec_chunk]        # truncate first
if args.action_repeat > 1:
    action_chunk = np.repeat(action_chunk, args.action_repeat, axis=0)

for action_row in action_chunk:
    obs, _, done, _ = env.step(action_row.tolist())
```

It works unchanged whether `action_chunk` came back with one row (OpenVLA)
or ten (UniVLA), which is exactly why this is the temporal condition that
is comparable across all of them.

**To add a fourth backbone: nothing to do.**

## ⚠️ Before running this on a new benchmark: is the action space relative?

> **Who this is for:** whoever sets up a benchmark this method has not been
> run on yet. **When:** once, at setup — not per run.
>
> Already checked: **LIBERO** (robosuite `OSC_POSE`) and **SimplerEnv** are
> both relative, so results from them are interpretable as-is.
> **Not checked: CALVIN**, which exposes *both* relative and absolute action
> modes, with the active one depending on the config and on how the policy
> was trained.

Repeating an action does something completely different depending on what
an action *is*:

| action space | an action means | repeating it twice |
|---|---|---|
| **relative / delta** (`Δx, Δy, Δz, Δrot, gripper`) | "move 5 cm forward" | moves 10 cm — the arm travels **twice as far** open-loop. The intervention as intended. |
| **absolute** (target joint angles or end-effector pose) | "go to position (30, 20)" | already there — the second step is a **no-op** and the arm holds still. |

### Why this is worth ten minutes

In an absolute action space the condition **costs nothing and does nothing**:

* success rate barely moves — no real intervention was applied
* model calls genuinely halve — the model really was called half as often

The results table then reads **"2× faster at no cost in accuracy"**, which
looks like the strongest result in the whole study and is entirely an
artifact of the controller absorbing the repeat. Nothing about the run
errors, warns, or looks wrong.

### The check

Hold one action for several steps and watch whether the arm keeps moving.
No theory needed.

```python
obs = env.reset()
a = policy.step(get_image(obs), instruction)[0]   # one action

positions = []
for _ in range(10):
    obs, *_ = env.step(list(a))                  # the SAME action, 10 times
    positions.append(read_ee_position(obs))      # however the env exposes it

print(positions)
# keeps drifting  -> relative   run this condition; results are meaningful
# stops after one -> absolute   do NOT run it; repeat is a no-op here
```

If the action space turns out to be absolute, the temporal axis needs a
different intervention on that benchmark — not this one.

## What it costs the trajectory (relative action spaces)

A repeated action doubles the displacement commanded before the policy sees
a new frame, so the arm travels twice as far open-loop between corrections.
The failure mode is therefore *overshoot on approach and imprecision at
contact*, not a uniform degradation — which is why it hurts tasks needing
fine placement far more than coarse reaching.

In [ ]:
# A policy tracking a target with proportional control, under repeat=1 vs 2.
# The gain is stable when each action is applied once; repeating it doubles
# the effective gain, which is what pushes the loop past 1.0 and overshoots.
def simulate(repeat, steps=40, gain=0.6, target=1.0):
    pos, trace = 0.0, []
    t = 0
    while t < steps:
        action = gain * (target - pos)          # one model call
        for _ in range(repeat):                 # executed `repeat` times
            pos += action
            trace.append(pos)
            t += 1
            if t >= steps:
                break
    return trace

a, b = simulate(1), simulate(2)
print(f"repeat=1  final {a[-1]:.3f}   max overshoot {max(a) - 1.0:+.3f}")
print(f"repeat=2  final {b[-1]:.3f}   max overshoot {max(b) - 1.0:+.3f}")
print("\nBoth converge here because the target does not move. On a real "
      "task\nthe overshoot lands the gripper past the object, and the "
      "correction\narrives one full call late.")

try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(7, 3.2))
    plt.axhline(1.0, color="gray", ls="--", lw=1, label="target")
    plt.plot(a, label="repeat=1")
    plt.plot(b, label="repeat=2")
    plt.xlabel("environment step"); plt.ylabel("position")
    plt.legend(); plt.tight_layout(); plt.show()
except ImportError:
    pass

## Measuring it correctly

**The cost of one model call does not change.** What halves is the number
of calls. So the per-call figure is identical to baseline and reporting it
alone would suggest the intervention did nothing.

Report either:

* **calls per episode** (halved), or
* **ms per environment step** = `model_ms_per_infer × calls / env_steps` (halved),

and say which. The `run_episode` in notebook 01 returns both
`model_ms_per_infer` and `model_ms_per_env_step` for exactly this reason.

One more asymmetry, and it is structural rather than empirical: a policy
that already emits a chunk of 10 and executes all of them is *already* amortised
10× per call. Applying repeat=2 on top pushes it to 20 environment steps of
open-loop execution between observations, while the same flag on a
single-action policy pushes it to 2.

**So repeat=2 is not one intervention strength across backbones.** The
quantity that actually determines the damage is *environment steps executed
per observation*, and repeat multiplies whatever the policy already does:

| | steps/obs at baseline | at repeat=2 |
|---|---|---|
| single-action policy | 1 | 2 |
| chunk-10 policy | 10 | **20** |

If the goal is to compare backbones at matched open-loop horizon rather
than at a matched flag value, record steps-per-observation alongside the
success rate — otherwise a backbone can look fragile when it was simply
pushed twice as far.

## Check the hook does what it claims

Self-contained: a stub env and a stub policy, no simulator. What it pins
down is that repeat halves the **calls** while leaving the number of
environment steps and the per-call cost alone — the exact confusion the
section above warns about.

In [ ]:
class _CountingEnv:
    """Counts env steps; never terminates, so the step cap decides."""

    def __init__(self, size=32):
        self.size, self.n = size, 0

    def reset(self):
        self.n = 0
        return {"cam": np.zeros((self.size, self.size, 3), np.uint8)}

    def step(self, action):
        self.n += 1
        return {"cam": np.zeros((self.size, self.size, 3), np.uint8)}, 0.0, False, {}


class _ChunkPolicy:
    def __init__(self, chunk):
        self.chunk = chunk

    def reset(self):
        pass

    def step(self, image, instruction):
        return np.zeros((self.chunk, 7), dtype=np.float32)


def _drive(chunk, repeat, max_steps=80):
    """Minimal driver, so this cell does not depend on notebook 01."""
    env, pol = _CountingEnv(), _ChunkPolicy(chunk)
    hook = make_action_repeat_hook(repeat=repeat)
    obs, steps, calls = env.reset(), 0, 0
    while steps < max_steps:
        actions = hook(np.asarray(pol.step(obs["cam"], "task")), {})
        calls += 1
        for row in actions:
            obs, *_ = env.step(row)
            steps += 1
            if steps >= max_steps:
                break
    return steps, calls


print(f"{'policy':<12}{'repeat':>7}{'env steps':>11}{'calls':>7}{'steps/call':>12}")
for chunk in (1, 10):
    base_steps, base_calls = _drive(chunk, 1)
    rep_steps, rep_calls = _drive(chunk, 2)
    for label, (st, ca) in [("baseline", (base_steps, base_calls)),
                            ("repeat 2", (rep_steps, rep_calls))]:
        print(f"chunk-{chunk:<7}{label:>7}{st:>11}{ca:>7}{st / ca:>12.1f}")
    assert rep_calls * 2 == base_calls, "repeat=2 must halve the calls"
    assert rep_steps == base_steps, "env steps must be unchanged"

print("\ncalls halve; env steps do not. The per-call cost is untouched,")
print("so reporting ms/call alone would show no effect at all.")

## Appendix — what we observed on our own runs

Context only. **None of this is a property of the method** — it is what
happened on the backbones and benchmarks we ran, at 50–96 episodes per
condition depending on the benchmark. At those sizes a difference of
roughly 10 points or less is not reliably distinguishable from chance.
Do not carry these numbers to a new setup — carry the questions.

| condition | observation |
|---|---|
| repeat 2, single-action policy | −8 points, not distinguishable from chance |
| repeat 2, chunk-10 policy | **−68 points** |
| chunk-exec k=2, one chunking backbone | **+13.6 points at 1.9× faster** |
| chunk-exec k=2, another chunking backbone | −12.5 points |

Rows 1 and 2 are the same flag at very different open-loop horizons (2 vs
20 env steps per observation), so they are not two measurements of one
intervention strength. Rows 3 and 4 are the same intervention with opposite
signs on two backbones that both support it.

The transferable lesson is the bookkeeping, not the numbers: **record
env-steps-per-observation next to every temporal result**, or a backbone
can look fragile when it was simply pushed ten times further.